In [15]:
import pandas as pd
df=pd.read_csv("forest_fire_labeled_v2.csv")
df.head()

,temperature,humidity,MQ2_raw,MQ135_raw,flame_intensity,light_intensity,soil_moisture,label
0,22.1,49.8,636,424,836,2151.3,27.4,0
1,21.7,73.6,528,426,1000,1976.1,37.8,0
2,30.2,26.0,533,556,796,4676.0,52.2,1
3,28.1,41.2,636,747,455,2783.4,35.5,1
4,31.0,27.3,679,529,870,1700.7,32.9,1


In [16]:
# Finding the maximum in MQ135 raw data
df.MQ135_raw.cummax()

0        424
1        426
2        556
3        747
4        747
        ... 
5235    1015
5236    1015
5237    1015
5238    1015
5239    1015
Name: MQ135_raw, Length: 5240, dtype: int64

In [17]:
df.isnull().sum

<bound method DataFrame.sum of       temperature  humidity  MQ2_raw  MQ135_raw  flame_intensity  \
0           False     False    False      False            False   
1           False     False    False      False            False   
2           False     False    False      False            False   
3           False     False    False      False            False   
4           False     False    False      False            False   
...           ...       ...      ...        ...              ...   
5235        False     False    False      False            False   
5236        False     False    False      False            False   
5237        False     False    False      False            False   
5238        False     False    False      False            False   
5239        False     False    False      False            False   

      light_intensity  soil_moisture  label  
0               False          False  False  
1               False          False  False  
2             

In [18]:
df.describe()

,temperature,humidity,MQ2_raw,MQ135_raw,flame_intensity,light_intensity,soil_moisture,label
count,5240.000000,5240.000000,5240.000000,5240.000000,5240.000000,5240.00000,5240.000000,5240.000000
mean,35.430802,52.354943,590.592176,476.444847,758.517557,3160.38521,46.860859,0.318893
std,14.592485,22.880722,132.611813,123.766278,223.243070,3133.90254,20.186905,0.466092
min,10.000000,0.000000,299.000000,293.000000,0.000000,0.00000,0.000000,0.000000
25%,25.000000,34.000000,498.000000,395.000000,672.000000,883.57500,31.700000,0.000000
50%,31.900000,53.900000,552.000000,431.000000,831.000000,2344.55000,46.400000,0.000000
75%,43.100000,70.000000,654.000000,520.000000,913.000000,4308.57500,60.825000,1.000000
max,89.200000,100.000000,1023.000000,1015.000000,1023.000000,19910.80000,99.700000,1.000000


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5240 entries, 0 to 5239
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   temperature      5240 non-null   float64
 1   humidity         5240 non-null   float64
 2   MQ2_raw          5240 non-null   int64  
 3   MQ135_raw        5240 non-null   int64  
 4   flame_intensity  5240 non-null   int64  
 5   light_intensity  5240 non-null   float64
 6   soil_moisture    5240 non-null   float64
 7   label            5240 non-null   int64  
dtypes: float64(4), int64(4)
memory usage: 327.6 KB


In [20]:
X=df.drop(columns=['label'],axis=1)
y=df['label']

In [21]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,AdaBoostClassifier,StackingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score, precision_score, f1_score,balanced_accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, fbeta_score
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
numeric_features=['temperature', 'humidity', 'MQ2_raw', 'MQ135_raw', 'flame_intensity', 'light_intensity', 'soil_moisture']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())])
model_pipeline_xgb = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[
        ('num', numeric_transformer, numeric_features)
    ])),
    ('classifier', XGBClassifier(use_label_encoder=False, eval_metric='mlogloss',learning_rate=0.01, n_estimators=50, max_depth=5))
])
#applying the model pipeline to the training data
model_pipeline_xgb.fit(X_train, y_train)
#applying the grid search to find the best hyperparameters for the model
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.1, 0.2]
}
f2_scorer = make_scorer(fbeta_score, beta=2, average='weighted')
grid_search = GridSearchCV(model_pipeline_xgb, param_grid, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search.fit(X_train, y_train)


c:\Users\LOQ\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\core.py:158: UserWarning: [18:46:53] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\LOQ\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\core.py:158: UserWarning: [18:47:06] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


,estimator,"Pipeline(step...=None, ...))])"
,param_grid,"{'classifier__learning_rate': [0.01, 0.1, ...], 'classifier__max_depth': [3, 5, ...], 'classifier__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...)]"


In [22]:
#evaluating the model on the test set
y_pred = grid_search.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred))


Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.98      0.97       720
           1       0.95      0.91      0.93       328

    accuracy                           0.96      1048
   macro avg       0.95      0.95      0.95      1048
weighted avg       0.96      0.96      0.96      1048

Confusion Matrix:
 [[703  17]
 [ 28 300]]
Balanced Accuracy Score: 0.9455115176151762


In [23]:
# training by AdaBoostClassifier
model_pipeline_ada = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features)])),
    ('classifier', AdaBoostClassifier(n_estimators=200, learning_rate=0.1, random_state=42,algorithm='deprecated'))
])
model_pipeline_ada.fit(X_train, y_train)
#finding the best hyperparameters for AdaBoostClassifier using grid search
param_grid_ada = {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__learning_rate': [0.01, 0.1, 1.0]
}
grid_search_ada = GridSearchCV(model_pipeline_ada, param_grid_ada, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search_ada.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'classifier__learning_rate': [0.01, 0.1, ...], 'classifier__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...)]"


In [24]:
#evaluating the AdaBoostClassifier model on the test set
y_pred_ada = grid_search_ada.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_ada))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_ada))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred_ada))

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.98      0.96       720
           1       0.95      0.89      0.92       328

    accuracy                           0.95      1048
   macro avg       0.95      0.93      0.94      1048
weighted avg       0.95      0.95      0.95      1048

Confusion Matrix:
 [[704  16]
 [ 36 292]]
Balanced Accuracy Score: 0.9340108401084011


In [25]:
#training with GradientBoostingClassifier
model_pipeline_gb = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features)])),
    ('classifier', GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42))
])
model_pipeline_gb.fit(X_train, y_train)
param_grid_gb = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__learning_rate': [0.01, 0.1, 0.2],
    'classifier__max_depth': [3, 5, 7],
    
}
grid_search_gb = GridSearchCV(model_pipeline_gb, param_grid_gb, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search_gb.fit(X_train, y_train)


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'classifier__learning_rate': [0.01, 0.1, ...], 'classifier__max_depth': [3, 5, ...], 'classifier__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...)]"


In [26]:
#evaluating the gradient boosting model on the test set
y_pred_gb = grid_search_gb.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_gb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_gb))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred_gb))


Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.97      0.97       720
           1       0.94      0.90      0.92       328

    accuracy                           0.95      1048
   macro avg       0.95      0.94      0.94      1048
weighted avg       0.95      0.95      0.95      1048

Confusion Matrix:
 [[702  18]
 [ 32 296]]
Balanced Accuracy Score: 0.938719512195122


In [27]:
#training with random forest classifier
model_pipeline_rf = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features)])),
    ('classifier', RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42))
])
model_pipeline_rf.fit(X_train, y_train)
param_grid_rf = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [3, 5, 7],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
}
grid_search_rf = GridSearchCV(model_pipeline_rf, param_grid_rf, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search_rf.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'classifier__max_depth': [3, 5, ...], 'classifier__min_samples_leaf': [1, 2, ...], 'classifier__min_samples_split': [2, 5, ...], 'classifier__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...)]"


In [28]:
#evalauting the random forest model on the test set
y_pred_rf = grid_search_rf.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred_rf))


Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.98      0.97       720
           1       0.95      0.91      0.93       328

    accuracy                           0.96      1048
   macro avg       0.95      0.94      0.95      1048
weighted avg       0.96      0.96      0.96      1048

Confusion Matrix:
 [[704  16]
 [ 29 299]]
Balanced Accuracy Score: 0.9446815718157182


In [29]:
#now training by StackingClassifier
base_estimators = [
    ('ada', AdaBoostClassifier(n_estimators=200, learning_rate=0.1, random_state=42,algorithm='deprecated')),
    ('gb', GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)),
    ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='mlogloss',learning_rate=0.01, n_estimators=50, max_depth=5))
]
meta_estimator = RandomForestClassifier(max_depth=7, n_estimators=50, min_samples_split=5, min_samples_leaf=2,random_state=42)
stacking_model = StackingClassifier(estimators=base_estimators, final_estimator=meta_estimator, cv=5, n_jobs=-1)
stacking_model.fit(X_train, y_train)
param_grid_stacking = {
    'final_estimator__n_estimators': [50, 100, 200],
    'final_estimator__max_depth': [3, 5, 7],
}
grid_search_stacking = GridSearchCV(stacking_model, param_grid_stacking, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search_stacking.fit(X_train, y_train)

,estimator,StackingClass... n_jobs=-1)
,param_grid,"{'final_estimator__max_depth': [3, 5, ...], 'final_estimator__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,estimator,None


In [30]:
#evaluating the stacking model on the test set
y_pred_stacking = grid_search_stacking.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_stacking))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_stacking))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred_stacking))


Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.98      0.97       720
           1       0.95      0.91      0.93       328

    accuracy                           0.95      1048
   macro avg       0.95      0.94      0.95      1048
weighted avg       0.95      0.95      0.95      1048

Confusion Matrix:
 [[703  17]
 [ 31 297]]
Balanced Accuracy Score: 0.9409383468834689


In [31]:
from sklearn.metrics import recall_score

model_results = {
    "XGBoost": grid_search,
    "AdaBoost": grid_search_ada,
    "GradientBoosting": grid_search_gb,
    "RandomForest": grid_search_rf,
    "StackingClassifier": grid_search_stacking,
}

print("Recall (class 1 = rain) comparison across all trained models:")
recalls = {}
for name, fitted_search in model_results.items():
    preds = fitted_search.predict(X_test)
    rec = recall_score(y_test, preds, pos_label=1)
    recalls[name] = rec
    print(f"  {name:20s} recall = {rec:.4f}")

best_model_name = max(recalls, key=recalls.get)
best_model = model_results[best_model_name]
print(f"\n>>> BEST MODEL BY RECALL: {best_model_name} ({recalls[best_model_name]:.4f})")
print("NOTE: if StackingClassifier does NOT win here, that's a real and reportable")
print("      result -- stacking doesn't always beat its own base models.")

Recall (class 1 = rain) comparison across all trained models:
  XGBoost              recall = 0.9146
  AdaBoost             recall = 0.8902
  GradientBoosting     recall = 0.9024
  RandomForest         recall = 0.9116
  StackingClassifier   recall = 0.9055

>>> BEST MODEL BY RECALL: XGBoost (0.9146)
NOTE: if StackingClassifier does NOT win here, that's a real and reportable
      result -- stacking doesn't always beat its own base models.


In [32]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve
import numpy as np

# Get out-of-fold probabilities on the TRAINING set only
oof_probs = cross_val_predict(
    best_model.best_estimator_, X_train, y_train,
    cv=5, method="predict_proba", n_jobs=-1
)[:, 1]   

precisions, recalls_curve, thresholds = precision_recall_curve(y_train, oof_probs, pos_label=1)

TARGET_RECALL = 0.9   
valid_idx = np.where(recalls_curve[:-1] >= TARGET_RECALL)[0]

if len(valid_idx) > 0:
    best_idx = valid_idx[np.argmax(precisions[valid_idx])]
    best_threshold = thresholds[best_idx]
    print(f"Chosen threshold (from train-set OOF probabilities): {best_threshold:.3f}")
    print(f"Expected recall={recalls_curve[best_idx]:.3f}, precision={precisions[best_idx]:.3f}")
else:
    print(f"No threshold reaches {TARGET_RECALL} recall on OOF predictions.")
    print("Consider lowering TARGET_RECALL, e.g. to 0.85.")
    best_threshold = 0.5

Chosen threshold (from train-set OOF probabilities): 0.288
Expected recall=0.900, precision=0.919


In [33]:
from sklearn.metrics import classification_report, confusion_matrix, fbeta_score

test_probs = best_model.predict_proba(X_test)[:, 1]
test_preds_thresholded = (test_probs >= best_threshold).astype(int)

print(f"\n=== FINAL: {best_model_name} @ threshold={best_threshold:.3f} on held-out TEST set ===")
print(classification_report(y_test, test_preds_thresholded, target_names=["no_rain", "rain"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, test_preds_thresholded))
print(f"F2-score: {fbeta_score(y_test, test_preds_thresholded, beta=2, pos_label=1):.4f}")

print(f"\n>>> threshold = {best_threshold:.3f} with model = {best_model_name}")
print(">>> in ESP32 HAZARD_MODELS config for rainfall prediction.")


=== FINAL: XGBoost @ threshold=0.288 on held-out TEST set ===
              precision    recall  f1-score   support

     no_rain       0.97      0.96      0.96       720
        rain       0.91      0.93      0.92       328

    accuracy                           0.95      1048
   macro avg       0.94      0.94      0.94      1048
weighted avg       0.95      0.95      0.95      1048

Confusion matrix:
[[689  31]
 [ 23 305]]
F2-score: 0.9254

>>> threshold = 0.288 with model = XGBoost
>>> in ESP32 HAZARD_MODELS config for rainfall prediction.


In [34]:
# distillation process
teacher_model = best_model.best_estimator_
teacher_model.fit(X_train, y_train)

c:\Users\LOQ\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\core.py:158: UserWarning: [18:54:36] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [35]:
soft_labels = teacher_model.predict_proba(X_train)

In [36]:
import tensorflow as tf
from tensorflow.keras import layers, models
student_model = models.Sequential([
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(soft_labels.shape[1], activation='softmax')
])
student_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
student_model.fit(X_train, soft_labels, epochs=50, batch_size=32, validation_split=0.2)

Epoch 1/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8354 - loss: 27.7496 - val_accuracy: 0.9642 - val_loss: 1.9179
Epoch 2/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9102 - loss: 2.2692 - val_accuracy: 0.9368 - val_loss: 1.2775
Epoch 3/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9096 - loss: 1.2383 - val_accuracy: 0.9225 - val_loss: 1.2847
Epoch 4/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9052 - loss: 1.7238 - val_accuracy: 0.9535 - val_loss: 1.0144
Epoch 5/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9153 - loss: 1.1536 - val_accuracy: 0.8892 - val_loss: 1.2239
Epoch 6/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8923 - loss: 2.5709 - val_accuracy: 0.8343 - val_loss: 2.8968
Epoch 7/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9314 - loss: 1.5539 - val_accuracy: 0.9583 - val_loss: 0.8206
Epoch 8/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9305 - loss: 1.0393 - val_accuracy: 0

In [37]:
convertor=tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_model = convertor.convert()

with open("forest_fire_predict_model.tflite", "wb") as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\LOQ\AppData\Local\Temp\tmpmdtg9k86\assets


INFO:tensorflow:Assets written to: C:\Users\LOQ\AppData\Local\Temp\tmpmdtg9k86\assets


Saved artifact at 'C:\Users\LOQ\AppData\Local\Temp\tmpmdtg9k86'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 7), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  2221181233712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2221181238640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2221181538496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2221181540784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2221181542016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2221181543248: TensorSpec(shape=(), dtype=tf.resource, name=None)
